In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fe4f7a6e-d91c-48c7-9a29-ecc5abef795c;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 448ms :: artifacts dl 25ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
df_reviews = spark.read.option('delimiter', ',') \
    .option('header', 'true') \
    .option('sep', ',') \
    .option('nullValue', 'NULL') \
    .option('inferSchema', 'true') \
    .option('multiLine', 'true') \
    .option('quote', '"') \
    .option('escape', '"') \
    .csv('s3a://last-mile-optimization-raw/dataset-orders/olist_order_reviews_dataset.csv')

df_reviews.show()

26/04/04 22:30:04 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                null|                  null| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                null|                  null| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                null|                  null| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                null|  Recebi bem antes ...| 2017-04-21 00:00:00|   

In [4]:
from pyspark.sql.functions import col

# Verificar se há order_ids nulos
null_order_id_count = df_reviews.filter(col("order_id").isNull()).count()

if null_order_id_count > 0:
    print(f"Foram encontrados {null_order_id_count} order_ids nulos.")
    df_reviews.filter(col("order_id").isNull()).show()
else:
    print("Não há order_ids nulos no DataFrame.")

Não há order_ids nulos no DataFrame.


In [5]:
from pyspark.sql.functions import col

# Contar o número de linhas antes da remoção
initial_count = df_reviews.count()

# Remover as linhas onde order_id é nulo
df_reviews = df_reviews.filter(col("order_id").isNotNull())

# Contar o número de linhas após a remoção
final_count = df_reviews.count()

removed_count = initial_count - final_count

if removed_count > 0:
    print(f"Foram removidas {removed_count} linhas com order_id nulo.")
    print(f"O DataFrame agora tem {final_count} linhas.")
else:
    print("Nenhuma linha com order_id nulo foi encontrada para remover.")


Nenhuma linha com order_id nulo foi encontrada para remover.


In [6]:
from pyspark.sql.functions import col, when, avg, concat_ws, collect_list

df_reviews = df_reviews.drop("review_id")
df_reviews = df_reviews.drop("review_comment_title")
df_reviews = df_reviews.drop("review_creation_date")
df_reviews = df_reviews.drop("review_answer_timestamp")


# Preencher valores nulos nos comentários com uma string vazia
df_reviews = df_reviews.withColumn(
    "review_comment_message",
    when(col("review_comment_message").isNull(), "").otherwise(col("review_comment_message"))
)
df_reviews.show()

# Agrupar por order_id, calcular a média do review_score e concatenar os comentários
df_reviews = df_reviews.groupBy("order_id").agg(
    avg("review_score").alias("review_score"),
    concat_ws(" | ", collect_list(when(col("review_comment_message") != "", col("review_comment_message")))).alias("review_comment_message")
)

df_reviews.show()


+--------------------+------------+----------------------+
|            order_id|review_score|review_comment_message|
+--------------------+------------+----------------------+
|73fc7af87114b3971...|           4|                      |
|a548910a1c6147796...|           5|                      |
|f9e4b658b201a9f2e...|           5|                      |
|658677c97b385a9be...|           5|  Recebi bem antes ...|
|8e6bfb81e283fa7e4...|           5|  Parabéns lojas la...|
|b18dcdf73be663668...|           1|                      |
|e48aa0d2dcec3a2e8...|           5|                      |
|c31a859e34e3adac2...|           5|                      |
|9c214ac970e842735...|           5|                      |
|b9bf720beb4ab3728...|           4|  aparelho eficient...|
|cdf9aa68e72324eeb...|           5|                      |
|3d374c9e46530bb5e...|           5|                      |
|9d6f15f95d01e79bd...|           4|  Mas um pouco ,tra...|
|2eaf8e099d871cd5c...|           4|                     

+--------------------+------------+----------------------+
|            order_id|review_score|review_comment_message|
+--------------------+------------+----------------------+
|00018f77f2f0320c5...|         4.0|                      |
|000229ec398224ef6...|         5.0|  Chegou antes do p...|
|00024acbcdf0a6daa...|         4.0|                      |
|00042b26cf59d7ce6...|         5.0|  Gostei pois veio ...|
|00054e8431b9d7675...|         4.0|                      |
|0005a1a1728c9d785...|         1.0|  Na descrição do p...|
|0005f50442cb953dc...|         4.0|                      |
|00063b381e2406b52...|         5.0|  Fiquei um pouco t...|
|0006ec9db01a64e59...|         5.0|  Excelente serviço...|
|0008288aa423d2a3f...|         5.0|                      |
|0009792311464db53...|         5.0|                      |
|000aed2e25dbad2f9...|         1.0|  Mudo minha opiniã...|
|000c3e6612759851c...|         5.0|  Recebi td certinh...|
|000e906b789b55f64...|         3.0|                     

In [7]:
df_reviews.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-orders/reviews_cleaned_dataset.csv')

spark.stop()

26/04/04 22:30:28 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/04 22:30:29 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
